# rotation-matrix-3d-y-axis — ex5: rotate a batch of points around Y

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d-y-axis`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five rotation-matrix patterns that ramp from `cos/sin` → matrix assembly → rotate-a-vector → composition law → rotate-a-batch. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import math
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `rotation-matrix-3d-y-axis`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d-y-axis"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Y-axis rotation — quick refresher

**The matrix.** For a right-hand rotation by `θ` about the Y axis:
```
R_y(θ) = [[ cos θ,  0,  sin θ],
          [ 0,      1,  0    ],
          [-sin θ,  0,  cos θ]]
```

**Why the middle row is `[0, 1, 0]`.** The Y axis is the rotation axis — anything along Y stays where it is. The X-Z plane is what gets rotated.

**Right-hand convention.** Looking down the +Y axis, the rotation goes counter-clockwise: +X → -Z, +Z → +X.

**Acting on vectors.**
- Column-vector: `v' = R @ v` (input shape `(3,)`).
- Batch of row-vectors: `points' = points @ R.T` (input shape `(N, 3)`).

**Composition.** `R_y(α) @ R_y(β) = R_y(α + β)` — rotations about a single axis add angles.

### Exercise 5 — rotate a batch of points around Y

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize matrix construction + matmul-shape arithmetic to rotate a (N, 3) batch of points around the Y axis in one operation.
> Keywords: batch-rotation, matmul-shape, ray-tracing, multi-kc
> ```

**KCs targeted:** `rotation-matrix-y-construct`, `rotation-applied-to-vector`, `rotate-batch-of-points`

Implement `ex5_rotate_batch(points, theta)`. Given a `(N, 3)` batch of 3-D points and a scalar angle `theta`, return a `(N, 3)` batch of rotated points.

Strategy: build `R_y(θ)` of shape `(3, 3)` once, then multiply with the batch. Two equivalent ways:
- `points @ R.T` — points are row vectors; transpose R to match.
- `(R @ points.T).T` — explicit column-vector form.

Pick whichever you find clearer. Both produce identical results.

> ⚠️ **Integrative exercise.** Combines 3 KCs (matrix construction, matvec → batched matmul shape arithmetic, batch rotation). Empirical work (Lohr et al. ITiCSE 2025) shows 3-concept exercises drop to ~40% solvability — expect a step up vs Exercises 1-4.

In [ ]:
def ex5_rotate_batch(points: Tensor, theta: Tensor) -> Tensor:
    """Rotate a (N, 3) batch of points around Y by theta. Returns (N, 3)."""
    raise NotImplementedError()


def _test_ex5():
    import math
    half_pi = t.tensor(math.pi / 2)
    points = t.tensor([
        [1.0, 0.0, 0.0],   # +X axis
        [0.0, 0.0, 1.0],   # +Z axis
        [0.0, 5.0, 0.0],   # pure Y (must be fixed)
        [1.0, 2.0, 0.0],   # XY corner
    ])
    out = ex5_rotate_batch(points, half_pi)
    assert out.shape == (4, 3), f'expected (4,3), got {out.shape}'
    expected = t.tensor([
        [0.0, 0.0, -1.0],  # X → -Z
        [1.0, 0.0, 0.0],   # +Z → +X
        [0.0, 5.0, 0.0],   # Y fixed
        [0.0, 2.0, -1.0],  # X→-Z, Y fixed
    ])
    assert t.allclose(out, expected, atol=1e-6), f'value mismatch:\n{out}\nvs\n{expected}'

    # Length-preserving check on a random batch.
    t.manual_seed(42)
    rnd = t.randn(8, 3)
    norms_before = rnd.pow(2).sum(dim=1).sqrt()
    norms_after = ex5_rotate_batch(rnd, t.tensor(0.7)).pow(2).sum(dim=1).sqrt()
    assert t.allclose(norms_before, norms_after, atol=1e-5), 'rotation must preserve per-row norms'

    # theta=0 identity.
    assert t.allclose(ex5_rotate_batch(points, t.tensor(0.0)), points, atol=1e-6)
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_rotate_batch(points: Tensor, theta: Tensor) -> Tensor:
    c = t.cos(theta).item()
    s = t.sin(theta).item()
    R = t.tensor([
        [c, 0.0, s],
        [0.0, 1.0, 0.0],
        [-s, 0.0, c],
    ])
    return points @ R.T
```

**Why `points @ R.T`?** PyTorch stores batches as `(N, D)` — rows are samples. The rotation formula `v' = R @ v` assumes `v` is a *column* vector. Two ways to reconcile:
- Stack as rows, transpose R: `points_row @ R.T` gives `(N, 3)` directly.
- Stack as cols, then transpose result: `(R @ points.T).T` — same math.

Both compile to the same matmul; pick by readability. `@ R.T` is the idiomatic PyTorch form because it preserves the `(N, D)` row-major layout.

**Where you'll use this.** Camera orbiting, object turntables, constructing per-vertex normals after rotating a mesh, generating rotated test data for invariance checks. Anywhere a scene needs to look at the model from a different angle.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex5',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()